In [10]:
print("XXX")

XXX


In [11]:
from email.header import decode_header

def decode_mime_header(s):
    """エンコードされたヘッダー（ファイル名など）をデコードする"""
    if not s: return ""
    parts = decode_header(s)
    decoded_parts = []
    for payload, charset in parts:
        if isinstance(payload, bytes):
            decoded_parts.append(payload.decode(charset or "utf-8", errors="replace"))
        else:
            decoded_parts.append(payload)
    return "".join(decoded_parts)


In [12]:
def get_total_bytes(exclude_extensions, file_infos, part):
    raw_filename = part.get_filename()
    total_bytes = 0
    if raw_filename:
        filename = decode_mime_header(raw_filename)
        if not any(filename.lower().endswith(ext) for ext in exclude_extensions):
                            # 添付ファイルのバイナリデータを取得してサイズを計算
            payload = part.get_payload(decode=True)
            if payload:
                file_size = len(payload)
                total_bytes += file_size
                # file_infos.append(f"{filename} ({file_size / 1024:.2f} KB)")
                file_infos.append(f"{filename} ({round(file_size / 1024, 2)} KB)")
    return total_bytes


In [13]:
import mailbox
import csv

def extract_attachments_with_size(mbox_path, output_csv):
    mbox = mailbox.mbox(mbox_path)
    exclude_extensions = ['.p7s']
    
    with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        # ヘッダーに「Total_Size_KB」と「File_Details」を追加
        writer.writerow(['Date', 'From', 'Subject', 'Total_Size_KB', 'File_Details'])

        sum_total_bytes = 0
        for message in mbox:
            file_infos = []
            total_bytes = 0
            if message.is_multipart():
                for part in message.walk():
                    content_disposition = str(part.get("Content-Disposition"))
                    is_attchment = "attachment" in content_disposition
                    is_inline = "inline" in content_disposition
                    is_sum = is_attchment or is_inline
                    if is_sum: total_bytes += get_total_bytes(exclude_extensions, file_infos, part)
            
            if file_infos:
                writer.writerow([
                    decode_mime_header(message['Date']),
                    decode_mime_header(message['From']),
                    decode_mime_header(message['Subject']),
                    round(total_bytes / 1024, 2), # 合計サイズ(KB)
                    ", ".join(file_infos)         # 各ファイル名とサイズ
                ])
            
            sum_total_bytes += total_bytes

    print(f"完了！容量計算済みのリストを {output_csv} に保存しました。")
    print(f"合計サイズ: {round(sum_total_bytes / 1024, 2)} KB")
    return sum_total_bytes



# 実行
# file = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files/2020.mbox"
# input_directory = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files"
# output_directory = "/home/yutaka/src_p/260508_Google_mail/output_y"
# extract_attachments_with_size(file, "attachments_size_list.csv")


In [16]:
import os
import glob

def process_all_mboxes(input_dir, output_dir, summary_writer):
    # 1) output_y フォルダが存在しない場合は作成
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # 3) 指定ディレクトリ配下の .mbox ファイルをすべて取得
    # 4) 4桁の年.mbox (例: 2020.mbox) にマッチするものを探す
    mbox_files = glob.glob(os.path.join(input_dir, "[0-9][0-9][0-9][0-9].mbox"))

    if not mbox_files:
        print("No matching .mbox files found.")
        return

    mbox_total_bytes =0
    for file_path in mbox_files:
        # ファイル名（2020.mboxなど）を取得
        base_name = os.path.basename(file_path)
        
        # 2) csvファイル名を設定（例: 2020.mbox.csv または 2020.csv）
        # ここでは 2020.mbox.csv となるように設定しています
        # csv_filename = f"{base_name}.csv"
        csv_filename = base_name.replace('.mbox', '.csv')
        output_path = os.path.join(output_dir, csv_filename)

        print(f"Processing: {base_name} -> {output_path}")
        
        # 既存の関数を実行
        year_total_bytes =0
        try:
            year_total_bytes = extract_attachments_with_size(file_path, output_path)
        except Exception as e:
            print(f"Error processing {base_name}: {e}")

        mbox_total_bytes += year_total_bytes

        summary_writer.writerow([base_name.replace('.mbox', ''), round(year_total_bytes / 1024 / 1024, 2), round(mbox_total_bytes / 1024 / 1024, 2)])  # 年とサイズ(MB)をsummaryに書き込む

    print(f"完了！合計サイズ: {round(mbox_total_bytes / 1024 / 1024, 2)} MB")

# --- 実行セクション ---
input_directory = "/home/yutaka/src_p/260506_Google_mail/split_mbox_files"
output_directory = "/home/yutaka/src_p/260508_Google_mail/output_y"
summery_file ="attachments_size_list_summary_y.csv"

with open(summery_file, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['Year', 'Total_Size_MB','Sum_total_Size_MB'])  # ヘッダー行
    process_all_mboxes(input_directory, output_directory,writer)


Processing: 2021.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2021.csv
完了！容量計算済みのリストを /home/yutaka/src_p/260508_Google_mail/output_y/2021.csv に保存しました。
合計サイズ: 4979.75 KB
Processing: 2019.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2019.csv
完了！容量計算済みのリストを /home/yutaka/src_p/260508_Google_mail/output_y/2019.csv に保存しました。
合計サイズ: 39803.23 KB
Processing: 2022.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2022.csv
完了！容量計算済みのリストを /home/yutaka/src_p/260508_Google_mail/output_y/2022.csv に保存しました。
合計サイズ: 1555.02 KB
Processing: 2026.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2026.csv
完了！容量計算済みのリストを /home/yutaka/src_p/260508_Google_mail/output_y/2026.csv に保存しました。
合計サイズ: 48.11 KB
Processing: 2024.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2024.csv
完了！容量計算済みのリストを /home/yutaka/src_p/260508_Google_mail/output_y/2024.csv に保存しました。
合計サイズ: 8623.17 KB
Processing: 2016.mbox -> /home/yutaka/src_p/260508_Google_mail/output_y/2016.csv
完了！容量計算済みのリストを /home/yuta